# Schema Matching — End-to-End Walkthrough

This notebook walks through the full pipeline for neural schema matching:
given two tables with different column naming conventions, predict which columns represent the same concept.

**Example:**

| Table A | Table B | Match? |
|---|---|---|
| `salary` | `income` | ✓ |
| `hire_date` | `start_date` | ✓ |
| `salary` | `start_date` | ✗ |

---
**Sections**
1. Data overview
2. Feature extraction
3. String-similarity baseline
4. Model training
5. Threshold optimisation
6. Evaluation & comparison
7. Error analysis
8. Predictions on custom column pairs

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('../').resolve()))

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

from src.preprocessing import column_to_text
from src.dataset import build_full_dataset, ColumnMatchingDataset, TABLE_PAIRS
from src.model import ColumnMatcher, get_device
from src.train import train_model
from src.evaluate import (
    evaluate_name_baseline,
    get_predictions,
    find_best_threshold,
    compute_metrics,
    compute_pr_auc,
    print_confusion_details,
)

print(f'Device: {get_device()}')

---
## 1. Data Overview

The dataset consists of **10 synthetic table pairs** across different business domains.
Each pair shares the same underlying data, but uses different column names.

In [ ]:
# Show one example table pair
df_a = pd.read_csv('../data/raw/employees_raw_a.csv')
df_b = pd.read_csv('../data/raw/employees_raw_b.csv')

print('Table A (employees_a)')
display(df_a)
print('Table B (employees_b)')
display(df_b)

In [ ]:
# Label distribution across all domains
labels_df = pd.read_csv('../data/labels/column_matches.csv')

domain_stats = (
    labels_df
    .assign(domain=labels_df['table_a'].str.replace('_a', '', regex=False))
    .groupby(['domain', 'label'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: 'negative', 1: 'positive'})
)

fig, ax = plt.subplots(figsize=(10, 4))
domain_stats.plot(kind='bar', ax=ax, color=['#d9534f', '#5cb85c'], width=0.6)
ax.set_title('Label distribution per domain', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Number of pairs')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(title='Label')
plt.tight_layout()
plt.show()

n_pos = int(labels_df['label'].sum())
n_neg = len(labels_df) - n_pos
print(f'Total: {len(labels_df)} pairs  |  {n_pos} positive  |  {n_neg} negative  |  ratio 1:{n_neg // n_pos}')

---
## 2. Feature Extraction

Each column is converted into a text string that captures:
- Column name
- Data type
- Null rate & unique ratio
- Min / max / mean (numeric columns only)
- Sample values

This richer representation lets the sentence encoder pick up on semantic signals beyond just the column name.

In [ ]:
rows = []
for col in df_a.columns:
    rows.append({'column': col, 'text representation': column_to_text(df_a, col)})

pd.set_option('display.max_colwidth', None)
display(pd.DataFrame(rows))

---
## 3. String-Similarity Baseline

Before training a model, we establish a simple baseline: compare column names using
[SequenceMatcher](https://docs.python.org/3/library/difflib.html) and predict a match if similarity > 0.5.

This baseline is fast and interpretable, but fails for semantically similar names that look different (`salary` vs. `income`).

In [ ]:
baseline_df = evaluate_name_baseline(labels_df)

bl_preds = baseline_df['prediction'].values
bl_labels = baseline_df['label'].values

bl_acc = accuracy_score(bl_labels, bl_preds)
bl_f1  = f1_score(bl_labels, bl_preds, zero_division=0)

print(f'Baseline — Accuracy: {bl_acc:.3f}  |  F1: {bl_f1:.3f}\n')
print('Cases where the baseline fails (FN: missed matches):')
missed = baseline_df[(baseline_df['label'] == 1) & (baseline_df['prediction'] == 0)]
display(missed[['column_a', 'column_b', 'similarity']].reset_index(drop=True))

The baseline misses semantically equivalent but lexically different pairs — exactly the cases where a learned model can add value.

---
## 4. Model Training

Architecture: a **Siamese network** built on `all-MiniLM-L6-v2` (384-dim embeddings).
Both columns are encoded independently, the embeddings are concatenated, and a small MLP predicts match probability.

```
column_a → SentenceTransformer → emb_a ─┐
                                          ├─ concat(768) → Linear→ReLU→Dropout
column_b → SentenceTransformer → emb_b ─┘              → Linear→ReLU→Dropout → logit
```

Training details:
- `BCEWithLogitsLoss` with `pos_weight=4` to counter the 1:4 class imbalance
- Adam optimizer + `ReduceLROnPlateau` scheduler
- Early stopping (patience=3) restoring best validation checkpoint

In [ ]:
full_df = build_full_dataset('../data/raw', labels_df)

train_df, val_df = train_test_split(
    full_df, test_size=0.2, stratify=full_df['label'], random_state=42
)

train_dataset = ColumnMatchingDataset(train_df.reset_index(drop=True))
val_dataset   = ColumnMatchingDataset(val_df.reset_index(drop=True))

pos_weight = (1 - full_df['label']).sum() / full_df['label'].sum()
print(f'Train: {len(train_dataset)}  |  Val: {len(val_dataset)}  |  pos_weight: {pos_weight:.1f}')

In [ ]:
model = train_model(
    train_dataset,
    val_dataset,
    epochs=10,
    lr=1e-3,
    batch_size=8,
    patience=3,
    pos_weight=pos_weight,
)

---
## 5. Threshold Optimisation

The default threshold of 0.5 is arbitrary. We scan the full range and pick the threshold that maximises F1 on the **validation set**.

In [ ]:
val_probs, val_labels = get_predictions(model, val_dataset)

thresholds = np.arange(0.1, 0.91, 0.01)
f1_scores  = [f1_score(val_labels, (val_probs >= t).astype(int), zero_division=0) for t in thresholds]

best_thresh, best_f1 = find_best_threshold(val_probs, val_labels)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_scores, linewidth=2)
ax.axvline(best_thresh, color='red', linestyle='--', label=f'Best threshold = {best_thresh:.2f}  (F1={best_f1:.3f})')
ax.set_xlabel('Threshold')
ax.set_ylabel('F1 Score')
ax.set_title('Validation F1 vs. Classification Threshold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Optimal threshold: {best_thresh:.2f}  →  Val F1: {best_f1:.3f}')

---
## 6. Evaluation & Comparison

We compare the neural model against the string-similarity baseline using the optimised threshold.

In [ ]:
train_probs, train_labels_arr = get_predictions(model, train_dataset)
train_metrics = compute_metrics(train_probs, train_labels_arr, threshold=best_thresh)
val_metrics   = compute_metrics(val_probs,   val_labels,       threshold=best_thresh)

results = pd.DataFrame([
    {'Method': 'String-Similarity Baseline', 'Split': 'all',   'Accuracy': bl_acc,                    'F1': bl_f1,                    'PR-AUC': '—'},
    {'Method': 'Neural Model',               'Split': 'train', 'Accuracy': train_metrics['accuracy'],  'F1': train_metrics['f1'],      'PR-AUC': f"{train_metrics['pr_auc']:.3f}"},
    {'Method': 'Neural Model',               'Split': 'val',   'Accuracy': val_metrics['accuracy'],    'F1': val_metrics['f1'],        'PR-AUC': f"{val_metrics['pr_auc']:.3f}"},
])

results[['Accuracy', 'F1']] = results[['Accuracy', 'F1']].applymap(lambda x: f'{x:.3f}' if isinstance(x, float) else x)
display(results.set_index(['Method', 'Split']))

In [ ]:
# Precision-Recall curves: model vs. baseline
from sklearn.metrics import precision_recall_curve

model_pr_auc, model_precision, model_recall = compute_pr_auc(val_probs, val_labels)

bl_val = baseline_df[baseline_df.index.isin(val_df.index)] if False else baseline_df
bl_precision, bl_recall, _ = precision_recall_curve(bl_labels, baseline_df['similarity'].values)
bl_pr_auc = float(np.trapz(bl_precision[::-1], bl_recall[::-1]))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(model_recall,  model_precision,  label=f'Neural model  (PR-AUC={model_pr_auc:.3f})', linewidth=2)
ax.plot(bl_recall,     bl_precision,     label=f'Baseline       (PR-AUC={bl_pr_auc:.3f})',   linewidth=2, linestyle='--')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Error Analysis

Which column pairs does the model get wrong? We inspect false positives and false negatives on the validation set.

In [ ]:
print_confusion_details(val_dataset, val_probs, val_labels, threshold=best_thresh)

---
## 8. Predictions on Custom Column Pairs

Run the trained model on arbitrary column pairs to see how it generalises.

In [ ]:
def predict_pair(model, col_name_a, values_a, col_name_b, values_b, threshold=best_thresh):
    """Predict whether two columns match given their names and sample values."""
    import pandas as pd
    df_a = pd.DataFrame({col_name_a: values_a})
    df_b = pd.DataFrame({col_name_b: values_b})
    text_a = column_to_text(df_a, col_name_a)
    text_b = column_to_text(df_b, col_name_b)
    model.eval()
    with torch.no_grad():
        logit = model([text_a], [text_b]).item()
        prob  = torch.sigmoid(torch.tensor(logit)).item()
    match = prob >= threshold
    print(f'  {col_name_a:20s}  ↔  {col_name_b:20s}  |  prob={prob:.3f}  |  {"MATCH" if match else "no match"}')


print('Custom predictions (threshold={:.2f}):'.format(best_thresh))
print()
predict_pair(model, 'salary',       [60000, 45000, 72000], 'income',          [60000, 45000, 72000])
predict_pair(model, 'dob',          ['1990-01-01', '1985-06-15'], 'birth_date', ['1990-01-01', '1985-06-15'])
predict_pair(model, 'customer_id',  [1, 2, 3],              'order_id',        [101, 102, 103])
predict_pair(model, 'phone_number', ['+49301234567'],        'email',           ['a@b.com'])
predict_pair(model, 'zip_code',     ['10115', '80331'],      'postal_code',     ['10115', '80331'])
predict_pair(model, 'revenue',      [1000, 2000, 3000],      'cost',            [500, 800, 1200])